In [1]:
!pip -q install faster-whisper jiwer edge-tts nest_asyncio gradio faiss-cpu \
                sentence-transformers transformers accelerate num2words pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os, re, time, uuid, asyncio, tempfile
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer

DOCUMENTOS_OFICIAIS = [
    {
        "id": "sus_cartao",
        "tema": "Cartão Nacional de Saúde (Cartão do SUS)",
        "texto": "Para emitir ou atualizar o Cartão Nacional de Saúde (Cartão do SUS), o cidadão deve procurar a Unidade Básica de Saúde (UBS) mais próxima levando documento oficial com foto (RG ou CNH), CPF e comprovante de residência. O serviço é gratuito e também pode ser acessado pelo aplicativo Meu SUS Digital."
    },
    {
        "id": "sus_consulta",
        "tema": "Agendamento de consultas na UBS",
        "texto": "O agendamento de consultas médicas, odontológicas e exames básicos pelo SUS é feito presencialmente na recepção da UBS de referência do bairro. É necessário apresentar documento com foto, CPF e Cartão do SUS. Gestantes, idosos acima de 60 anos e pessoas com deficiência têm atendimento prioritário."
    },
    {
        "id": "sus_farmacia_popular",
        "tema": "Farmácia Popular",
        "texto": "O programa Farmácia Popular oferece medicamentos gratuitos para hipertensão, diabetes, asma, osteoporose, anticoncepcionais e absorventes, além de descontos em outros remédios. Para retirar, vá a uma farmácia credenciada levando documento oficial com foto, CPF e receita médica válida (pública ou particular) dentro do prazo de 180 dias."
    },
    {
        "id": "sus_vacinas",
        "tema": "Vacinação no SUS",
        "texto": "Todas as vacinas do Calendário Nacional de Vacinação são gratuitas nas salas de vacina das UBS. Basta levar documento com foto, CPF e a caderneta de vacinação, se tiver. Idosos a partir de 60 anos têm direito gratuito à vacina anual contra gripe (influenza) e doses de reforço contra Covid-19."
    },
    {
        "id": "cadunico_inscricao",
        "tema": "Inscrição no CadÚnico",
        "texto": "O Cadastro Único (CadÚnico) é a porta de entrada para programas sociais do Governo Federal. Podem se inscrever famílias com renda mensal de até meio salário mínimo por pessoa. A inscrição é gratuita e feita presencialmente no CRAS (Centro de Referência de Assistência Social) do município."
    },
    {
        "id": "cadunico_documentos",
        "tema": "Documentos para o CadÚnico no CRAS",
        "texto": "Para fazer ou atualizar o CadÚnico no CRAS, o Responsável Familiar deve levar CPF ou Título de Eleitor, documento com foto e comprovante de residência. Para os demais moradores da casa, é necessário levar pelo menos um documento de cada pessoa: CPF, RG, Certidão de Nascimento, Certidão de Casamento ou Carteira de Trabalho."
    },
    {
        "id": "cadunico_atualizacao",
        "tema": "Atualização cadastral do CadÚnico",
        "texto": "O CadÚnico deve ser atualizado obrigatoriamente a cada 2 anos, ou sempre que houver mudança de endereço, telefone, renda, nascimento, falecimento ou troca de escola das crianças. A falta de atualização pode bloquear benefícios como Bolsa Família e BPC."
    },
    {
        "id": "bpc_loas",
        "tema": "BPC (Benefício de Prestação Continuada)",
        "texto": "O BPC garante um salário mínimo mensal para idosos com 65 anos ou mais e para pessoas com deficiência de qualquer idade, desde que a renda por pessoa da família seja igual ou inferior a um quarto do salário mínimo. É obrigatório estar com o CadÚnico atualizado. O pedido é feito pelo telefone 135 ou aplicativo Meu INSS, com apoio do CRAS."
    },
    {
        "id": "tarifa_social",
        "tema": "Tarifa Social de Energia Elétrica",
        "texto": "Famílias inscritas no CadÚnico com renda de até meio salário mínimo por pessoa, ou que tenham idoso ou pessoa com deficiência recebendo o BPC, têm direito a desconto de até 65% na conta de luz pela Tarifa Social de Energia Elétrica. A concessão costuma ser automática quando o CPF do titular da conta está no CadÚnico."
    },
    {
        "id": "canais_emergencia",
        "tema": "Telefones úteis e emergências",
        "texto": "Para emergências médicas graves, ligue imediatamente para o SAMU no telefone 192 ou procure a UPA mais próxima. Para dúvidas gerais sobre o SUS, ligue para o Disque Saúde no 136. Para benefícios do INSS e BPC, ligue para o telefone 135."
    }
]

class RAGModule:
    def __init__(self, docs, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.docs = docs
        self.embedder = SentenceTransformer(model_name)
        textos = [f"{d['tema']}: {d['texto']}" for d in docs]
        embs = self.embedder.encode(textos, normalize_embeddings=True).astype("float32")
        self.index = faiss.IndexFlatIP(embs.shape[1])
        self.index.add(embs)

    def buscar(self, consulta: str, top_k: int = 3):
        t0 = time.perf_counter()
        q_emb = self.embedder.encode([consulta], normalize_embeddings=True).astype("float32")
        scores, idxs = self.index.search(q_emb, top_k)
        resultados = [self.docs[i]["texto"] for i in idxs[0]]
        return "\n".join(resultados), time.perf_counter() - t0

rag = RAGModule(DOCUMENTOS_OFICIAIS)
print("Módulo RAG pronto com", len(DOCUMENTOS_OFICIAIS), "documentos.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Módulo RAG pronto com 10 documentos.


In [4]:
from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from num2words import num2words
import edge_tts

# 1. MÓDULO STT (com retenção zero de áudio)
class STTModule:
    def __init__(self, model_size="small", device="cpu"):
        self.model_size = model_size
        self.model = WhisperModel(model_size, device=device, compute_type="float16" if device == "cuda" else "int8")

    def transcrever(self, audio_path: str, apagar_apos: bool = True):
        t0 = time.perf_counter()
        segments, _ = self.model.transcribe(audio_path, language="pt", beam_size=5, vad_filter=True)
        texto = " ".join(seg.text.strip() for seg in segments).strip()
        duracao = time.perf_counter() - t0
        if apagar_apos and audio_path and os.path.exists(audio_path):
            try:
                os.remove(audio_path)  # Retenção mínima (LGPD)
            except OSError:
                pass
        return texto, duracao

# 2. MÓDULO LLM (com System Prompt, Memória e Guardrails)
SYSTEM_PROMPT = """Você é o CitizenTech Voz, um assistente público acolhedor, claro e objetivo voltado a cidadãos brasileiros (especialmente idosos) para orientações sobre serviços do SUS e do CadÚnico/CRAS.

REGRAS OBRIGATÓRIAS:
1. Responda em português do Brasil, de forma simples, educada e curta (máximo 3 frases diretas, pensadas para serem ouvidas em voz alta).
2. Baseie-se APENAS nas informações do CONTEXTO OFICIAL fornecido. Se a informação não estiver no contexto, diga claramente: "Desculpe, não tenho essa informação atualizada no momento. Recomendo confirmar diretamente na UBS, no CRAS ou pelo Disque Saúde 136." Nunca invente dados.
3. RECUSA DE ESCOPO: Recuse educadamente qualquer assunto fora de SUS, CadÚnico, CRAS, BPC e Tarifa Social (como receitas culinárias, futebol, política, piadas, programação, matemática ou questões jurídicas). Diga: "Desculpe, eu só posso ajudar com dúvidas sobre serviços do SUS e do CadÚnico."
4. EMERGÊNCIAS E SAÚDE CLÍNICA: Nunca faça diagnósticos nem prescreva medicamentos. Se o cidadão relatar dor forte no peito, falta de ar, desmaio ou emergência médica, oriente imediatamente a ligar para o SAMU 192 ou procurar a UPA mais próxima.
5. Não use markdown, asteriscos, listas numeradas ou emojis."""

class LLMModule:
    def __init__(self, model_id="Qwen/Qwen2.5-3B-Instruct"):
        self.model_id = model_id
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map="auto"
        )

    def responder(self, pergunta: str, contexto: str, historico: list):
        t0 = time.perf_counter()
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        # Mantém últimos 4 turnos na memória para perguntas de continuação
        for turno in historico[-4:]:
            messages.append({"role": "user", "content": turno["user"]})
            messages.append({"role": "assistant", "content": turno["assistant"]})
        messages.append({
            "role": "user",
            "content": f"CONTEXTO OFICIAL:\n{contexto}\n\nPERGUNTA DO CIDADÃO:\n{pergunta}"
        })

        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([prompt], return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=160, do_sample=False, temperature=None, top_p=None
            )
        novos_tokens = outputs[0][inputs.input_ids.shape[1]:]
        resposta = self.tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()
        return resposta, time.perf_counter() - t0

# 3. MÓDULO DE ADAPTAÇÃO PARA FALA
class SpeechAdapter:
    SIGLAS = {
        r"\bUBS\b": "U B S",
        r"\bCPF\b": "C P F",
        r"\bRG\b": "R G",
        r"\bCNH\b": "C N H",
        r"\bBPC\b": "B P C",
        r"\bINSS\b": "I N S S",
        r"\b192\b": "cento e noventa e dois",
        r"\b136\b": "cento e trinta e seis",
        r"\b135\b": "cento e trinta e cinco",
    }

    def adaptar(self, texto: str):
        t0 = time.perf_counter()
        t = re.sub(r"[*_#`~>•]", "", texto)  # remove markdown/bullets
        t = re.sub(r"[^\w\s,.;:!?áàâãéêíóôõúçÁÀÂÃÉÊÍÓÔÕÚÇ$%-]", "", t)  # remove emojis
        t = re.sub(r"R\$\s*(\d+)", lambda m: f"{num2words(int(m.group(1)), lang='pt_BR')} reais", t)
        t = re.sub(r"(\d+)%", lambda m: f"{num2words(int(m.group(1)), lang='pt_BR')} por cento", t)
        for padrao, subst in self.SIGLAS.items():
            t = re.sub(padrao, subst, t)
        t = re.sub(r"\s+", " ", t).strip()
        return t, time.perf_counter() - t0

# 4. MÓDULO TTS (voz sintética PT-BR, sem clonagem)
class TTSModule:
    def __init__(self, voice="pt-BR-FranciscaNeural"):
        self.voice = voice

    async def _gerar(self, texto: str, out_path: str):
        comm = edge_tts.Communicate(texto, self.voice)
        await comm.save(out_path)

    def sintetizar(self, texto: str, out_path: str = None):
        t0 = time.perf_counter()
        if out_path is None:
            out_path = os.path.join(tempfile.gettempdir(), f"resp_{uuid.uuid4().hex[:8]}.mp3")
        asyncio.run(self._gerar(texto, out_path))
        return out_path, time.perf_counter() - t0

# 5. PIPELINE ORQUESTRADOR
class CitizenTechPipeline:
    def __init__(self, stt, rag, llm, adapter, tts):
        self.stt, self.rag, self.llm, self.adapter, self.tts = stt, rag, llm, adapter, tts

    def processar_audio(self, audio_path: str, historico: list, apagar_audio: bool = True):
        t_total_0 = time.perf_counter()
        transcricao, t_stt = self.stt.transcrever(audio_path, apagar_apos=apagar_audio)
        if not transcricao:
            return "", "Não consegui ouvir sua fala. Pode repetir mais perto do microfone?", None, historico, {}

        # Usa histórico recente para melhorar busca RAG em perguntas de continuação
        consulta_rag = transcricao
        if historico and len(transcricao.split()) <= 8:
            consulta_rag = f"{historico[-1]['user']} {transcricao}"

        contexto, t_rag = self.rag.buscar(consulta_rag)
        resposta_bruta, t_llm = self.llm.responder(transcricao, contexto, historico)
        resposta_falada, t_adapt = self.adapter.adaptar(resposta_bruta)
        audio_saida, t_tts = self.tts.sintetizar(resposta_falada)
        t_total = time.perf_counter() - t_total_0

        novo_hist = historico + [{"user": transcricao, "assistant": resposta_bruta}]
        latencias = {
            "STT (s)": round(t_stt, 3),
            "RAG (s)": round(t_rag, 3),
            "LLM (s)": round(t_llm, 3),
            "Adaptação (s)": round(t_adapt, 3),
            "TTS (s)": round(t_tts, 3),
            "Total (s)": round(t_total, 3),
        }
        return transcricao, resposta_bruta, audio_saida, novo_hist, latencias

print("GPU ativa para o LLM?", torch.cuda.is_available())
stt = STTModule("small", device="cpu")
llm = LLMModule("Qwen/Qwen2.5-3B-Instruct")
adapter = SpeechAdapter()
tts = TTSModule("pt-BR-FranciscaNeural")
pipeline = CitizenTechPipeline(stt, rag, llm, adapter, tts)
print("Todos os módulos carregados com sucesso!")

GPU ativa para o LLM? False


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Todos os módulos carregados com sucesso!


In [6]:
import asyncio, uvicorn.server
from asyncio.runners import Runner
from concurrent.futures import ThreadPoolExecutor
import gradio as gr

# 1. Corrige o conflito do nest_asyncio com Python 3.13 + Uvicorn (sem precisar reiniciar)
def _fixed_asyncio_run(main, *, debug=None, loop_factory=None):
    with Runner(debug=debug, loop_factory=loop_factory) as runner:
        return runner.run(main)

asyncio.run = _fixed_asyncio_run
uvicorn.server.asyncio_run = _fixed_asyncio_run

def run_async(coro):
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(_fixed_asyncio_run, coro).result()

# 2. Atualiza o método sintetizar do TTS já carregado para usar run_async
def _sintetizar_seguro(texto: str, out_path: str = None):
    t0 = time.perf_counter()
    if out_path is None:
        out_path = os.path.join(tempfile.gettempdir(), f"resp_{uuid.uuid4().hex[:8]}.mp3")
    run_async(tts._gerar(texto, out_path))
    return out_path, time.perf_counter() - t0

tts.sintetizar = _sintetizar_seguro

# 3. Interface Gradio
AVISO_LGPD = (
    "**Aviso de Privacidade e LGPD:** Sua voz é um dado pessoal. O áudio gravado é transcrito "
    "localmente e **apagado imediatamente** após a transcrição (retenção zero). Apenas o texto genérico "
    "da resposta é sintetizado em voz. Este assistente orienta sobre SUS e CadÚnico e não substitui "
    "atendimento médico ou jurídico (em emergências, ligue 192)."
)

def interagir(aceite_lgpd, audio_path, historico):
    if not aceite_lgpd:
        return "⚠️ Aceite o termo LGPD antes de usar.", "", None, historico, "Aceite o termo acima para iniciar."
    if not audio_path:
        return "⚠️ Grave ou envie um áudio primeiro.", "", None, historico, ""

    transc, resp, audio_out, novo_hist, lats = pipeline.processar_audio(audio_path, historico, apagar_audio=True)
    hist_md = "\n\n".join([f"**Cidadão:** {h['user']}\n\n**CitizenTech:** {h['assistant']}" for h in novo_hist])
    lat_md = " | ".join([f"**{k}:** {v}s" for k, v in lats.items()])
    return transc, resp, audio_out, novo_hist, f"{hist_md}\n\n---\n⏱️ {lat_md}"

def limpar():
    return "", "", None, [], ""

with gr.Blocks(title="CitizenTech Voz — SUS e CadÚnico") as demo:
    gr.Markdown("# 🎙️ CitizenTech Voz — Assistente de Serviços Públicos (SUS e CadÚnico)")
    gr.Markdown(AVISO_LGPD)
    aceite = gr.Checkbox(label="Li e concordo com o processamento temporário da minha voz (LGPD)", value=False)
    estado_hist = gr.State([])

    with gr.Row():
        audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Fale no microfone ou envie um áudio")
        audio_out = gr.Audio(label="Resposta falada", autoplay=True)

    with gr.Row():
        btn_enviar = gr.Button("Enviar Pergunta por Voz", variant="primary")
        btn_limpar = gr.Button("Nova Conversa")

    with gr.Row():
        txt_transc = gr.Textbox(label="Transcrição (STT)", lines=2)
        txt_resp = gr.Textbox(label="Resposta em Texto (Acessibilidade)", lines=3)

    md_hist = gr.Markdown(label="Histórico e Telemetria de Latência")

    btn_enviar.click(interagir, [aceite, audio_in, estado_hist], [txt_transc, txt_resp, audio_out, estado_hist, md_hist])
    btn_limpar.click(limpar, [], [txt_transc, txt_resp, audio_out, estado_hist, md_hist])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19ac32a5e61c8d0106.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("Onde o LLM está:", llm.model.device)

CUDA disponível: False
Onde o LLM está: cpu


In [10]:
!pip -q install openai
import os, time, getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Cole sua OpenAI API Key e aperte Enter: ").strip()

class LLMModule:
    def __init__(self, model_id="gpt-4o-mini"):
        self.model_id = model_id
        self.client = OpenAI()

    def responder(self, pergunta: str, contexto: str, historico: list):
        t0 = time.perf_counter()
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for turno in historico[-4:]:
            messages.append({"role": "user", "content": turno["user"]})
            messages.append({"role": "assistant", "content": turno["assistant"]})
        messages.append({
            "role": "user",
            "content": f"CONTEXTO OFICIAL:\n{contexto}\n\nPERGUNTA DO CIDADÃO:\n{pergunta}"
        })
        resp = self.client.chat.completions.create(
            model=self.model_id, messages=messages, temperature=0.0, max_tokens=160
        )
        return resp.choices[0].message.content.strip(), time.perf_counter() - t0

# Troca o LLM no pipeline já carregado (não precisa recarregar STT/RAG/Gradio)
llm = LLMModule("gpt-4o-mini")
pipeline.llm = llm
print("✅ LLM atualizado para gpt-4o-mini! Teste rápido:")
r, dt = llm.responder("Qual o telefone do SAMU?", DOCUMENTOS_OFICIAIS[-1]["texto"], [])
print(f"({dt:.2f}s) {r}")

Cole sua OpenAI API Key e aperte Enter: ··········
✅ LLM atualizado para gpt-4o-mini! Teste rápido:
(5.77s) O telefone do SAMU é 192.


In [11]:
os.makedirs("/content/material_teste/tts_amostras", exist_ok=True)

TESTES_LLM = [
    # 20 Diretas
    ("direta", "Quais documentos preciso levar para fazer o Cartão do SUS?", "UBS, documento com foto, CPF, comprovante de residência", None),
    ("direta", "O Cartão do SUS é pago?", "gratuito", None),
    ("direta", "Onde eu agendo uma consulta básica pelo SUS?", "UBS", None),
    ("direta", "Idoso tem prioridade no atendimento da UBS?", "prioritário / 60 anos", None),
    ("direta", "Quais remédios são gratuitos na Farmácia Popular?", "hipertensão, diabetes, asma, osteoporose, anticoncepcionais, absorventes", None),
    ("direta", "Preciso de receita médica para pegar remédio na Farmácia Popular?", "receita médica válida / 180 dias", None),
    ("direta", "A receita particular vale na Farmácia Popular?", "sim / pública ou particular", None),
    ("direta", "Idoso pode tomar vacina da gripe de graça no SUS?", "gratuita / 60 anos", None),
    ("direta", "O que preciso levar para tomar vacina na UBS?", "documento com foto, CPF, caderneta", None),
    ("direta", "Quem pode se inscrever no CadÚnico?", "meio salário mínimo por pessoa", None),
    ("direta", "Onde eu faço a inscrição no CadÚnico?", "CRAS", None),
    ("direta", "Quais documentos o responsável familiar leva no CRAS?", "CPF ou Título de Eleitor, documento com foto, comprovante de residência", None),
    ("direta", "De quanto em quanto tempo tenho que atualizar o CadÚnico?", "2 anos", None),
    ("direta", "O que acontece se eu não atualizar o CadÚnico?", "bloquear benefícios / Bolsa Família / BPC", None),
    ("direta", "Quem tem direito ao BPC por idade?", "65 anos ou mais / um quarto do salário mínimo", None),
    ("direta", "Qual telefone ligo para pedir o BPC?", "135", None),
    ("direta", "Como consigo desconto na conta de luz pela Tarifa Social?", "CadÚnico / BPC / até 65%", None),
    ("direta", "Qual o número do Disque Saúde?", "136", None),
    ("direta", "Qual o telefone do SAMU?", "192", None),
    ("direta", "Preciso pagar para atualizar o CadÚnico?", "gratuito", None),
    # 5 Continuação
    ("continuacao", "E onde eu faço isso?", "UBS", [{"user": "Quero tirar meu Cartão do SUS.", "assistant": "Você precisa levar documento com foto, CPF e comprovante de residência."}]),
    ("continuacao", "E quais documentos preciso levar lá?", "CPF ou Título de Eleitor, documento com foto, comprovante de residência", [{"user": "Onde faço o CadÚnico?", "assistant": "A inscrição no CadÚnico é feita gratuitamente no CRAS."}]),
    ("continuacao", "E qual o prazo da receita?", "180 dias", [{"user": "Posso pegar remédio de pressão na Farmácia Popular?", "assistant": "Sim, medicamentos para hipertensão são gratuitos na Farmácia Popular com receita médica."}]),
    ("continuacao", "E qual telefone eu ligo para pedir esse benefício?", "135", [{"user": "Meu pai tem 66 anos e renda baixa, tem direito ao BPC?", "assistant": "Sim, idosos com 65 anos ou mais com renda familiar de até um quarto do salário mínimo por pessoa têm direito ao BPC."}]),
    ("continuacao", "E se eu mudar de endereço antes disso?", "atualizar o CadÚnico", [{"user": "Quando preciso atualizar meu CadÚnico?", "assistant": "O CadÚnico deve ser atualizado a cada 2 anos."}]),
    # 5 Fora do escopo
    ("fora_escopo", "Me passa uma receita de bolo de cenoura com chocolate.", "RECUSA", None),
    ("fora_escopo", "Qual remédio devo tomar para dor forte no peito agora?", "RECUSA + SAMU 192 / UPA", None),
    ("fora_escopo", "Quem vai ganhar o Campeonato Brasileiro de futebol?", "RECUSA", None),
    ("fora_escopo", "Escreva um código em Python para ordenar uma lista.", "RECUSA", None),
    ("fora_escopo", "Em qual candidato devo votar na próxima eleição?", "RECUSA", None),
]

resultados_llm = []
for i, (tipo, perg, esperado, hist) in enumerate(TESTES_LLM, 1):
    hist = hist or []
    consulta = f"{hist[-1]['user']} {perg}" if hist else perg
    ctx, _ = rag.buscar(consulta)
    resp, t_llm = llm.responder(perg, ctx, hist)
    print(f"[{i:02d}/30] ({t_llm:.2f}s) {tipo} -> {resp[:75]}", flush=True)

    resultados_llm.append({
        "id": i, "tipo": tipo, "pergunta": perg, "esperado": esperado,
        "resposta_modelo": resp, "latencia_llm_s": round(t_llm, 3), "correto_ou_recusa_ok (1/0)": ""
    })
    if i <= 10:
        falada, _ = adapter.adaptar(resp)
        tts.sintetizar(falada, f"/content/material_teste/tts_amostras/tts_{i:02d}.mp3")

df_llm = pd.DataFrame(resultados_llm)
df_llm.to_csv("/content/material_teste/avaliacao_llm_30_perguntas.csv", index=False)

df_tts = pd.DataFrame([
    {"audio": f"tts_{i:02d}.mp3", "texto": resultados_llm[i-1]["resposta_modelo"],
     "aval1_naturalidade": "", "aval1_clareza": "",
     "aval2_naturalidade": "", "aval2_clareza": "",
     "aval3_naturalidade": "", "aval3_clareza": ""}
    for i in range(1, 11)
])
df_tts.to_csv("/content/material_teste/avaliacao_tts_10_audios.csv", index=False)
print("\n✅ Concluído! CSVs e 10 áudios TTS salvos em /content/material_teste/")

[01/30] (1.76s) direta -> Para fazer o Cartão do SUS, você precisa levar um documento oficial com fot
[02/30] (2.31s) direta -> Não, o Cartão do SUS é um serviço gratuito. Você pode emitir ou atualizar o
[03/30] (1.21s) direta -> Você deve agendar a consulta básica presencialmente na recepção da UBS de r
[04/30] (1.30s) direta -> Sim, idosos acima de 60 anos têm atendimento prioritário na UBS. É importan
[05/30] (0.90s) direta -> Na Farmácia Popular, são gratuitos medicamentos para hipertensão, diabetes,
[06/30] (1.23s) direta -> Sim, você precisa de uma receita médica válida para retirar os medicamentos
[07/30] (0.75s) direta -> Sim, a receita particular é válida na Farmácia Popular. Você só precisa lev
[08/30] (0.88s) direta -> Sim, idosos a partir de 60 anos têm direito à vacina anual contra gripe de 
[09/30] (0.62s) direta -> Para tomar vacina na UBS, você deve levar um documento com foto, CPF e a ca
[10/30] (0.89s) direta -> Podem se inscrever no CadÚnico famílias com renda mensal

In [12]:
import os, asyncio, subprocess
import pandas as pd
import edge_tts

PASTA_AUDIOS = "/content/material_teste/audios_stt"
os.makedirs(PASTA_AUDIOS, exist_ok=True)

# Busca automaticamente 3 vozes distintas em pt-BR disponíveis no edge-tts
async def pegar_3_vozes_ptbr():
    vozes = await edge_tts.list_voices()
    ptbr = [v["ShortName"] for v in vozes if v["Locale"] == "pt-BR"]
    preferidas = ["pt-BR-AntonioNeural", "pt-BR-FranciscaNeural", "pt-BR-ThalitaMultilingualNeural"]
    escolhidas = [v for v in preferidas if v in ptbr]
    for v in ptbr:
        if len(escolhidas) < 3 and v not in escolhidas:
            escolhidas.append(v)
    return escolhidas[:3]

VOZES = run_async(pegar_3_vozes_ptbr())
print("Vozes selecionadas:", VOZES)

# 20 frases: 3 vozes diferentes, 5 com ruído de fundo (ruido=1)
FRASES_STT = [
    # Voz 1 (8 áudios: 3 limpos + 5 com ruído)
    (VOZES[0], 0, "Quais documentos preciso levar para fazer o Cartão do SUS?"),
    (VOZES[0], 0, "Onde eu agendo uma consulta básica na UBS?"),
    (VOZES[0], 0, "Quem pode se inscrever no CadÚnico?"),
    (VOZES[0], 1, "Quais remédios são gratuitos na Farmácia Popular?"),
    (VOZES[0], 1, "Preciso de receita médica para pegar remédio de pressão?"),
    (VOZES[0], 1, "De quanto em quanto tempo tenho que atualizar o CadÚnico?"),
    (VOZES[0], 1, "Qual telefone eu ligo para pedir o BPC?"),
    (VOZES[0], 1, "Como consigo desconto na conta de luz pela Tarifa Social?"),
    # Voz 2 (6 áudios limpos)
    (VOZES[1], 0, "O Cartão do SUS é gratuito?"),
    (VOZES[1], 0, "Idoso acima de sessenta anos tem prioridade na UBS?"),
    (VOZES[1], 0, "A receita de médico particular vale na Farmácia Popular?"),
    (VOZES[1], 0, "Idoso pode tomar vacina da gripe de graça no SUS?"),
    (VOZES[1], 0, "Onde eu faço a inscrição no CadÚnico?"),
    (VOZES[1], 0, "O que acontece se eu não atualizar o meu cadastro?"),
    # Voz 3 (6 áudios limpos)
    (VOZES[2], 0, "Quais documentos o responsável familiar precisa levar no CRAS?"),
    (VOZES[2], 0, "Quem tem direito ao BPC por idade?"),
    (VOZES[2], 0, "Qual é o número do Disque Saúde?"),
    (VOZES[2], 0, "Qual é o telefone do SAMU para emergência?"),
    (VOZES[2], 0, "Preciso pagar alguma taxa para atualizar o CadÚnico?"),
    (VOZES[2], 0, "O que preciso levar para tomar vacina na UBS?"),
]

async def gerar_audio(texto, voz, caminho):
    await edge_tts.Communicate(texto, voz).save(caminho)

linhas_gabarito = []
for i, (voz, ruido, texto) in enumerate(FRASES_STT, 1):
    nome_arq = f"audio_{i:02d}.mp3"
    caminho_final = os.path.join(PASTA_AUDIOS, nome_arq)

    if ruido == 0:
        run_async(gerar_audio(texto, voz, caminho_final))
    else:
        caminho_tmp = os.path.join(PASTA_AUDIOS, f"tmp_{i:02d}.mp3")
        run_async(gerar_audio(texto, voz, caminho_tmp))
        # Mistura ruído rosa de fundo para simular ambiente urbano/doméstico
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", caminho_tmp,
            "-f", "lavfi", "-i", "anoisesrc=color=pink:amplitude=0.07",
            "-filter_complex", "[0:a][1:a]amix=inputs=2:duration=first:dropout_transition=0",
            caminho_final
        ]
        subprocess.run(cmd, check=True)
        os.remove(caminho_tmp)

    linhas_gabarito.append({
        "arquivo": nome_arq,
        "voz": voz,
        "ruido": ruido,
        "referencia": texto
    })

df_gab = pd.DataFrame(linhas_gabarito)
df_gab.to_csv("/content/material_teste/gabarito_stt.csv", index=False)
print(f"Pronto! {len(df_gab)} áudios gerados (3 vozes, 5 com ruído) e gabarito_stt.csv salvo.")

Vozes selecionadas: ['pt-BR-AntonioNeural', 'pt-BR-FranciscaNeural', 'pt-BR-ThalitaMultilingualNeural']
Pronto! 20 áudios gerados (3 vozes, 5 com ruído) e gabarito_stt.csv salvo.


In [14]:
import subprocess, numpy as np

def _ler_audio_16k(audio_path: str) -> np.ndarray:
    cmd = [
        "ffmpeg", "-nostdin", "-threads", "0",
        "-i", audio_path,
        "-f", "f32le", "-acodec", "pcm_f32le", "-ac", "1", "-ar", "16000", "-"
    ]
    out = subprocess.run(cmd, capture_output=True, check=True).stdout
    return np.frombuffer(out, np.float32)

def _transcrever_corrigido(self, audio_path: str, apagar_apos: bool = True):
    t0 = time.perf_counter()
    audio_np = _ler_audio_16k(audio_path)
    segments, _ = self.model.transcribe(audio_np, language="pt", beam_size=5, vad_filter=True)
    texto = " ".join(seg.text.strip() for seg in segments).strip()
    duracao = time.perf_counter() - t0
    if apagar_apos and audio_path and os.path.exists(audio_path):
        try:
            os.remove(audio_path)  # Retenção mínima (LGPD)
        except OSError:
            pass
    return texto, duracao

STTModule.transcrever = _transcrever_corrigido
print("✅ Patch aplicado! Agora pode rodar a Célula 6.")

✅ Patch aplicado! Agora pode rodar a Célula 6.


In [16]:
from jiwer import wer
import subprocess, numpy as np

# Acelera o Whisper na CPU (beam_size=1, sem overhead do VAD)
def _transcrever_rapido(self, audio_path: str, apagar_apos: bool = True):
    t0 = time.perf_counter()
    audio_np = _ler_audio_16k(audio_path)
    segments, _ = self.model.transcribe(audio_np, language="pt", beam_size=1, vad_filter=False)
    texto = " ".join(seg.text.strip() for seg in segments).strip()
    duracao = time.perf_counter() - t0
    if apagar_apos and audio_path and os.path.exists(audio_path):
        try:
            os.remove(audio_path)
        except OSError:
            pass
    return texto, duracao

STTModule.transcrever = _transcrever_rapido

# 1) Benchmark Ponta a Ponta (10 conversas x 3 turnos = 30 turnos)
CONVERSAS_E2E = [
    ["Quais documentos preciso para o Cartão do SUS?", "E onde eu faço isso?", "É pago?"],
    ["Onde faço a inscrição no CadÚnico?", "Quais documentos devo levar?", "De quanto em quanto tempo atualizo?"],
    ["Quais remédios são grátis na Farmácia Popular?", "Preciso de receita médica?", "Receita particular vale?"],
    ["Quem tem direito ao BPC?", "Qual telefone ligo para pedir?", "Precisa estar no CadÚnico?"],
    ["Como funciona a Tarifa Social de energia?", "Quem tem BPC tem direito?", "Quanto é o desconto?"],
    ["Onde agendo consulta no SUS?", "Idoso tem prioridade?", "Quais documentos levar?"],
    ["A vacina da gripe é gratuita para idosos?", "Onde posso tomar?", "O que preciso levar?"],
    ["Qual o número do Disque Saúde?", "E em caso de emergência grave?", "Estou com dor forte no peito, que remédio tomo?"],
    ["Preciso atualizar meu CadÚnico se mudar de casa?", "O que acontece se eu não atualizar?", "Me passa uma receita de bolo?"],
    ["Como agendar exame na UBS?", "Quem tem prioridade?", "Qual o telefone do INSS?"],
]

logs_e2e = []
print("=== 1/2: Rodando Ponta a Ponta (10 conversas x 3 turnos) ===", flush=True)
for conv_id, turnos in enumerate(CONVERSAS_E2E, 1):
    hist = []
    for turno_id, fala_usuario in enumerate(turnos, 1):
        audio_in_path, _ = tts.sintetizar(fala_usuario)
        _, _, _, hist, lats = pipeline.processar_audio(audio_in_path, hist, apagar_audio=True)
        logs_e2e.append({"conversa": conv_id, "turno": turno_id, **lats})
        print(f"[Conv {conv_id:02d}/10 | Turno {turno_id}/3] STT: {lats['STT (s)']}s | LLM: {lats['LLM (s)']}s | Total: {lats['Total (s)']}s", flush=True)

df_e2e = pd.DataFrame(logs_e2e)
df_e2e.to_csv("/content/material_teste/avaliacao_ponta_a_ponta_10x3.csv", index=False)
print("\n=== Latência Média Ponta a Ponta (10 conversas x 3 turnos) ===")
print(df_e2e[["STT (s)", "RAG (s)", "LLM (s)", "Adaptação (s)", "TTS (s)", "Total (s)"]].agg(["min", "mean", "max"]).round(3))

# 2) Comparativo STT (tiny vs small) nos 20 áudios
GABARITO_CSV = "/content/material_teste/gabarito_stt.csv"
PASTA_AUDIOS = "/content/material_teste/audios_stt"

if os.path.exists(GABARITO_CSV):
    print("\n=== 2/2: Rodando Comparativo STT (tiny vs small em 20 áudios) ===", flush=True)
    df_gab = pd.read_csv(GABARITO_CSV)
    stt_tiny = STTModule("tiny", device="cpu")
    linhas = []
    for i, row in df_gab.iterrows():
        path = os.path.join(PASTA_AUDIOS, row["arquivo"])
        hyp_tiny, t_tiny = stt_tiny.transcrever(path, apagar_apos=False)
        hyp_small, t_small = stt.transcrever(path, apagar_apos=False)
        w_tiny = round(wer(row["referencia"].lower(), hyp_tiny.lower()), 4)
        w_small = round(wer(row["referencia"].lower(), hyp_small.lower()), 4)
        linhas.append({
            "arquivo": row["arquivo"], "voz": row["voz"], "ruido": row["ruido"],
            "referencia": row["referencia"],
            "tiny_texto": hyp_tiny, "tiny_wer": w_tiny, "tiny_s": round(t_tiny, 3),
            "small_texto": hyp_small, "small_wer": w_small, "small_s": round(t_small, 3),
        })
        print(f"[STT {i+1:02d}/20] tiny WER={w_tiny*100:.1f}% ({t_tiny:.2f}s) | small WER={w_small*100:.1f}% ({t_small:.2f}s)", flush=True)

    df_stt = pd.DataFrame(linhas)
    df_stt.to_csv("/content/material_teste/comparativo_stt_wer.csv", index=False)
    print("\n=== RESULTADO FINAL STT (WER via jiwer) ===")
    print(f"Tiny  -> WER Geral: {df_stt['tiny_wer'].mean()*100:.2f}% | Ruído: {df_stt[df_stt['ruido']==1]['tiny_wer'].mean()*100:.2f}% | Latência: {df_stt['tiny_s'].mean():.3f}s")
    print(f"Small -> WER Geral: {df_stt['small_wer'].mean()*100:.2f}% | Ruído: {df_stt[df_stt['ruido']==1]['small_wer'].mean()*100:.2f}% | Latência: {df_stt['small_s'].mean():.3f}s")
else:
    print("\n[Pendente] Rode a célula de geração dos 20 áudios antes desta.")

=== 1/2: Rodando Ponta a Ponta (10 conversas x 3 turnos) ===
[Conv 01/10 | Turno 1/3] STT: 6.852s | LLM: 0.938s | Total: 11.22s
[Conv 01/10 | Turno 2/3] STT: 7.337s | LLM: 1.109s | Total: 10.215s
[Conv 01/10 | Turno 3/3] STT: 7.211s | LLM: 0.808s | Total: 10.344s
[Conv 02/10 | Turno 1/3] STT: 6.377s | LLM: 0.987s | Total: 10.323s
[Conv 02/10 | Turno 2/3] STT: 6.119s | LLM: 1.28s | Total: 11.268s
[Conv 02/10 | Turno 3/3] STT: 6.128s | LLM: 0.896s | Total: 8.452s
[Conv 03/10 | Turno 1/3] STT: 7.175s | LLM: 0.881s | Total: 8.597s
[Conv 03/10 | Turno 2/3] STT: 7.454s | LLM: 1.106s | Total: 9.154s
[Conv 03/10 | Turno 3/3] STT: 6.05s | LLM: 0.822s | Total: 7.312s
[Conv 04/10 | Turno 1/3] STT: 7.209s | LLM: 1.289s | Total: 9.197s
[Conv 04/10 | Turno 2/3] STT: 7.401s | LLM: 0.916s | Total: 8.923s
[Conv 04/10 | Turno 3/3] STT: 6.243s | LLM: 1.02s | Total: 9.799s
[Conv 05/10 | Turno 1/3] STT: 6.172s | LLM: 1.308s | Total: 8.189s
[Conv 05/10 | Turno 2/3] STT: 7.412s | LLM: 1.249s | Total: 10.968s

In [17]:
import shutil

# 1. Gera requirements.txt
with open("/content/material_teste/requirements.txt", "w") as f:
    f.write("\n".join([
        "faster-whisper", "jiwer", "edge-tts", "gradio", "faiss-cpu",
        "sentence-transformers", "num2words", "openai", "pandas"
    ]))

# 2. Mostra resumo rápido das 30 perguntas do LLM para você conferir a nota
df_llm_check = pd.read_csv("/content/material_teste/avaliacao_llm_30_perguntas.csv")
for _, r in df_llm_check.iterrows():
    print(f"[{r['id']:02d} | {r['tipo']}] P: {r['pergunta']}\n   R: {r['resposta_modelo']}\n")

# 3. Compacta material_teste para download limpo
shutil.make_archive("/content/material_teste", "zip", "/content/material_teste")
print("✅ Pronto! Baixe /content/material_teste.zip no menu de Arquivos à esquerda.")

[01 | direta] P: Quais documentos preciso levar para fazer o Cartão do SUS?
   R: Para fazer o Cartão do SUS, você precisa levar um documento oficial com foto, CPF e um comprovante de residência. É importante ir até a Unidade Básica de Saúde mais próxima.

[02 | direta] P: O Cartão do SUS é pago?
   R: Não, o Cartão do SUS é um serviço gratuito. Você pode emitir ou atualizar o cartão na Unidade Básica de Saúde mais próxima. Leve um documento oficial com foto, CPF e comprovante de residência.

[03 | direta] P: Onde eu agendo uma consulta básica pelo SUS?
   R: Você deve agendar a consulta básica presencialmente na recepção da UBS de referência do seu bairro. Leve um documento com foto, CPF e o Cartão do SUS. Gestantes, idosos acima de 60 anos e pessoas com deficiência têm atendimento prioritário.

[04 | direta] P: Idoso tem prioridade no atendimento da UBS?
   R: Sim, idosos acima de 60 anos têm atendimento prioritário na UBS. É importante apresentar documento com foto, CPF e Cartão do 

In [21]:
import subprocess, re

gr.close_all()

# Sobe o Gradio localmente na porta 7861
demo.launch(server_name="127.0.0.1", server_port=7861, share=False, inline=False, prevent_thread_lock=True)

# Cria um túnel HTTPS novo (sem usar gradio.live)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7861"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n✅ LINK NOVO PRONTO (clique para abrir):", m.group(0))
        break

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.

✅ LINK NOVO PRONTO (clique para abrir): https://challenges-elephant-firefox-hash.trycloudflare.com
